In [23]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk.corpus import wordnet
import pandas as pd
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import emoji

lemmatizer = WordNetLemmatizer()

nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [24]:
from tqdm import tqdm

tqdm.pandas()

appname = 'linkedin'
df = pd.read_csv(appname+'_reviews.csv')

# PREPROCESSING
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# diambil dari modul asah
def clean(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka
 
    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    return text

def case_folding(teks):
    return str(teks).lower()

def hapus_angka(teks):
    return ''.join([char for char in teks if not str(char).isdigit()])

def hapus_tanda_baca(teks):
    punctuation_set = set(string.punctuation)
    
    return ''.join([char for char in teks if char not in punctuation_set])

def hapus_whitespace(teks):
    return teks.strip()

negation_words = {'tidak', 'bukan', 'ga', 'gk', 'enggak', 'tak', 'tanpa', 'ngga', 'nga', 'tdak', 'tdk', 'g', 'gak', 'engga', 'kaga'}

def handle_negation(sentence):
    tokens = word_tokenize(sentence)
    processed_tokens = []
    skip_next = False
    for i, token in enumerate(tokens):
        if skip_next:
            skip_next = False
            continue
        if token in negation_words:
            if i + 1 < len(tokens):
                processed_tokens.append(tokens[i+1] + '_NEG')
                skip_next = True
        else:
            processed_tokens.append(token)
    return ' '.join(processed_tokens)

def hapus_stopwords(teks):
    tokenized_words = word_tokenize(teks)
    id_stopwords = set(stopwords.words('indonesian'))
    id_stopwords.update({'blom', 'bgt', 'jd', 'jdi', 'ja', 'dlu', 'y', 'yah', 'ol', 'bgus', 'yaa', 'dong', 'nya', 'masa', 'mau', 'rek', 'cb', 'dulu', 'babget', 'kb', 'moga', 'bngt', 'lbh', 'seep', 'gk', 'ga', 'g', 'nga', 'ngga', 'to', 'gsn', 'la', 'dul', 'cb', 'bgs', 'leh', 'ntr', 'aplikasi', 'akun', 'login', 'nya', 'yg', 'ga', 'gk', 'gak', 'min', 'kak', 'bang', 'sih', 'kok', 'deh', 'dong', 'nih', 'dll', 'trs', 'kalo', 'udh', 'udah', 'sy', 'saya', 'tdk', 'tdak', 'aja', 'apk', 'linkedin', 'gajelas', 'gue', 'gua' 'gw', 'bgt', 'gool', 'tyop', 'dlu', 'lu', 'lo', 'loe', 'tuk', 'bast', 'link', 'to', 'gsn', 'aplc', 'log', 'in', 'login', 'sy', 'sdh', 'blm', 'let', 'tes', 'wow', 'ajg', 'blok', 'utk', 'app', 'loginya', 'gak', 'd', 'zih', 'linkdin', 'ya', 'bintang', 'kasih', 'moga', 'orang', 'coba', 'bisa'})

    important_words = [kata for kata in tokenized_words if kata not in id_stopwords] # kata already lower
    
    return ' '.join(important_words)

def stem(teks):
    words = word_tokenize(teks)
    stemmer  = PorterStemmer()
    stemmed_words = [stemmer.stem(kata) for kata in words]

    return ' '.join(stemmed_words)

def stem_indo(teks):
    return stemmer.stem(teks)

def lemmatize(teks):
    words = word_tokenize(teks)
    lemmatized_words = [lemmatizer.lemmatize(kata, pos=wordnet.VERB) for kata in words]
    
    return ' '.join(lemmatized_words)

def demojize(teks):
    return emoji.demojize(teks, delimiters=(" ", " "))

# hapus baris dengan mising values / duplikat 
df = df.dropna()
df = df.drop_duplicates()

# aplikasikan fungsi ke kolom teks tiap baris
df['text'] = df['text'].progress_apply(clean)
df['text'] = df['text'].progress_apply(case_folding)
df['text'] = df['text'].progress_apply(hapus_angka)
df['text'] = df['text'].progress_apply(hapus_tanda_baca)
df['text'] = df['text'].progress_apply(hapus_whitespace)
df['text'] = df['text'].progress_apply(handle_negation)
df['text'] = df['text'].progress_apply(hapus_stopwords)
print(f"Starting stemming for {len(df)} rows...")
df['text'] = df['text'].progress_apply(stem_indo)
# df['text'] = df['text'].apply(demojize)
# df['text'] = df['text'].apply(lemmatize)
print(pd.Series(' '.join(df['text']).split()).value_counts()[:50])

df.to_csv(appname+'_preprocessed_reviews.csv', index=False)


100%|██████████| 18172/18172 [00:05<00:00, 3539.50it/s]


Starting stemming for 18172 rows...


100%|██████████| 18172/18172 [00:01<00:00, 12932.63it/s]

neg           3668
kerja         1671
bantu         1359
bisa          1358
bagus         1278
verifikasi    1027
cari           856
good           766
masuk          635
susah          605
tolong         591
mudah          521
manfaat        513
baik           500
banget         487
moga           472
guna           385
aman           367
the            367
i              366
kali           351
cepat          350
ok             344
and            344
informasi      337
buka           321
salah          309
daftar         309
periksa        305
mantap         303
jelas          303
job            294
for            291
info           271
lowong         270
ribet          270
gambar         270
aplikasi       266
mohon          264
usaha          253
email          248
ada            245
update         242
suka           242
bikin          235
sulit          227
loker          224
gabisa         223
terima         223
linkedln       219
Name: count, dtype: int64


In [25]:
import pandas as pd
import csv
import requests
from io import StringIO

In [26]:
df = pd.read_csv(appname+'_preprocessed_reviews.csv')

# Membaca data kamus kata-kata positif dari GitHub
lexicon_positive = dict()
 
response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv')
# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub
 
if response.status_code == 200:
    # Jika permintaan berhasil
    reader = csv.reader(StringIO(response.text), delimiter=',')
    # Membaca teks respons sebagai file CSV menggunakan pembaca CSV dengan pemisah koma
 
    for row in reader:
        # Mengulangi setiap baris dalam file CSV
        lexicon_positive[row[0]] = int(row[1])
        # Menambahkan kata-kata positif dan skornya ke dalam kamus lexicon_positive
else:
    print("Failed to fetch positive lexicon data")
 
# Membaca data kamus kata-kata negatif dari GitHub
lexicon_negative = dict()
 
response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv')
# Mengirim permintaan HTTP untuk mendapatkan file CSV dari GitHub
 
if response.status_code == 200:
    # Jika permintaan berhasil
    reader = csv.reader(StringIO(response.text), delimiter=',')
    # Membaca teks respons sebagai file CSV menggunakan pembaca CSV dengan pemisah koma
 
    for row in reader:
        # Mengulangi setiap baris dalam file CSV
        lexicon_negative[row[0]] = int(row[1])
        # Menambahkan kata-kata negatif dan skornya dalam kamus lexicon_negative
else:
    print("Failed to fetch negative lexicon data")


def sentiment_analysis(text):
    score = 0
    # Inisialisasi skor sentimen ke 0
 
    for word in str(text).split():
        # Mengulangi setiap kata dalam teks
 
        if (word in lexicon_positive):
            score = score + lexicon_positive[word]
        elif(word in lexicon_negative):
            score = score + lexicon_negative[word]
            # Jika kata ada dalam kamus positif, tambahkan skornya ke skor sentimen
 
    polarity=''
    # Inisialisasi variabel polaritas
 
    if (score > 0):
        polarity = 'positive'
        # Jika skor sentimen lebih besar atau sama dengan 0, maka polaritas adalah positif
    elif (score < 0):
        polarity = 'negative'
        # Jika skor sentimen kurang dari 0, maka polaritas adalah negatif
 
    else:
        polarity = 'neutral'
    # Ini adalah bagian yang bisa digunakan untuk menentukan polaritas netral jika diperlukan
 
    # print(f'{text}, score : {score}\n')
    return score, polarity
    # Mengembalikan skor sentimen dan polaritas teks

results = df['text'].apply(sentiment_analysis)

results = list(zip(*results))
df['polarity_score'] = results[0]
df['polarity'] = results[1]
print(df['polarity'].value_counts())

df.to_csv(appname+'_labeled_reviews.csv', index=False)

polarity
positive    9524
neutral     5358
negative    3290
Name: count, dtype: int64


In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import BernoulliNB

In [28]:
df = pd.read_csv(appname+'_labeled_reviews.csv')

X = df['text']
y = df['polarity']
 
# Ekstraksi fitur dengan TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000,max_df=0.8)
X_tfidf = tfidf.fit_transform(X.values.astype('U'))

# Konversi hasil ekstraksi fitur menjadi dataframe
features_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
 
# Menampilkan hasil ekstraksi fitur
# features_df
 
# Bagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Membuat objek model Naive Bayes (Bernoulli Naive Bayes)
naive_bayes = BernoulliNB()
 
# Melatih model Naive Bayes pada data pelatihan
naive_bayes.fit(X_train, y_train)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_nb = naive_bayes.predict(X_train.toarray())
y_pred_test_nb = naive_bayes.predict(X_test.toarray())
 
# Evaluasi akurasi model Naive Bayes
accuracy_train_nb = accuracy_score(y_pred_train_nb, y_train)
accuracy_test_nb = accuracy_score(y_pred_test_nb, y_test)
 
# Menampilkan akurasi
print('Naive Bayes - accuracy_train:', accuracy_train_nb)
print('Naive Bayes - accuracy_test:', accuracy_test_nb)

 
# Membuat objek model Random Forest
random_forest = RandomForestClassifier()
 
# Melatih model Random Forest pada data pelatihan
random_forest.fit(X_train.toarray(), y_train)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_rf = random_forest.predict(X_train.toarray())
y_pred_test_rf = random_forest.predict(X_test.toarray())
 
# Evaluasi akurasi model Random Forest
accuracy_train_rf = accuracy_score(y_pred_train_rf, y_train)
accuracy_test_rf = accuracy_score(y_pred_test_rf, y_test)
 
# Menampilkan akurasi
print('Random Forest - accuracy_train:', accuracy_train_rf)
print('Random Forest - accuracy_test:', accuracy_test_rf)

 
# Membuat objek model Logistic Regression
logistic_regression = LogisticRegression()
 
# Melatih model Logistic Regression pada data pelatihan
logistic_regression.fit(X_train.toarray(), y_train)
 
# Prediksi sentimen pada data pelatihan dan data uji
y_pred_train_lr = logistic_regression.predict(X_train.toarray())
y_pred_test_lr = logistic_regression.predict(X_test.toarray())
 
# Evaluasi akurasi model Logistic Regression pada data pelatihan
accuracy_train_lr = accuracy_score(y_pred_train_lr, y_train)
 
# Evaluasi akurasi model Logistic Regression pada data uji
accuracy_test_lr = accuracy_score(y_pred_test_lr, y_test)
 
# Menampilkan akurasi
print('Logistic Regression - accuracy_train:', accuracy_train_lr)
print('Logistic Regression - accuracy_test:', accuracy_test_lr)

Naive Bayes - accuracy_train: 0.8089702139368508
Naive Bayes - accuracy_test: 0.7823933975240716
Random Forest - accuracy_train: 0.993877691408131
Random Forest - accuracy_test: 0.8883081155433288
Logistic Regression - accuracy_train: 0.9431794730687212
Logistic Regression - accuracy_test: 0.8998624484181568


In [29]:
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_onehot = tf.keras.utils.to_categorical(y_encoded)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf.toarray(),
    y_onehot,
    test_size=0.2,
    random_state=42
)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(y_onehot.shape[1], activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

loss, acc = model.evaluate(X_test, y_test)
print(f"Fine-tuned Test accuracy: {acc:.4f}")
print(f"Fine-tuned Test loss: {loss:.4f}")


Epoch 1/200
455/455 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.7388 - loss: 0.6719 - val_accuracy: 0.8014 - val_loss: 0.6641
Epoch 2/200
455/455 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8896 - loss: 0.3477 - val_accuracy: 0.8922 - val_loss: 0.3407
Epoch 3/200
455/455 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9325 - loss: 0.2366 - val_accuracy: 0.8957 - val_loss: 0.3508
Epoch 4/200
455/455 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9529 - loss: 0.1811 - val_accuracy: 0.8993 - val_loss: 0.3670
Epoch 5/200
455/455 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9624 - loss: 0.1576 - val_accuracy: 0.9018 - val_loss: 0.3692
Epoch 6/200
455/455 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9691 - loss: 0.1414 - val_accuracy: 0.9029 - val_loss: 0.3863
Epoch 7/200
455/455 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9761 - loss: 0.1239 - val_accuracy: 0.9026 - val_loss: 0.3974
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8922 - loss: 0.3407
Fine-tuned Test accura

In [30]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc:.4f}")
print(f"Test loss: {loss:.4f}")

import numpy as np

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)


114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8922 - loss: 0.3407
Test accuracy: 0.8922
Test loss: 0.3407
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
